In [ ]:
#Pandas is required before using this notebook!
!pip install pandas

In [2]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

In [3]:
DATASET_CSV = "roboto2_dataset.csv"

## Example 1: Load the dataset

In [4]:
df = pd.read_csv(DATASET_CSV, keep_default_na=False, na_values=[])
print(f"Loaded {len(df):,} rows, columns: {list(df.columns)}")
df.head()

Loaded 620 rows, columns: ['paper_id', 'paper_parse', 'manual_assessment', 'roboto2_assessment']


,paper_id,paper_parse,manual_assessment,roboto2_assessment
0,29054,null,"[{""domain_risk_levels"": [""Some concerns"", ""Som...",[]
1,29057,null,"[{""domain_risk_levels"": [""Some concerns"", ""Low...",[]
2,29058,null,"[{""domain_risk_levels"": [""Some concerns"", ""Som...",[]
3,29059,null,"[{""domain_risk_levels"": [""Some concerns"", ""Som...",[]
4,29062,null,"[{""domain_risk_levels"": [""Some concerns"", ""Som...",[]


## Example 2: Count rows with a manual assessment

In [5]:
def has_non_empty_assessment(cell_value):
    assessment_list = json.loads(cell_value)
    return len(assessment_list) > 0

rows_with_manual_assessment = df["manual_assessment"].map(has_non_empty_assessment)
print(f"Rows with manual_assessment: {int(rows_with_manual_assessment.sum())}")

Rows with manual_assessment: 245


## Example 3: Count rows with a roboto2 assessment

In [6]:
def has_non_empty_assessment(cell_value):
    assessment_list = json.loads(cell_value)
    return len(assessment_list) > 0

rows_with_roboto2_assessment = df["roboto2_assessment"].map(has_non_empty_assessment)
print(f"Rows with roboto2_assessment: {int(rows_with_roboto2_assessment.sum())}")

Rows with roboto2_assessment: 203


## Example 4: Risk-assessment count per domain across all manual assessments

Each manual assessment carries `domain_risk_levels`, a 5-element list (one entry per domain 1–5). Tally levels per domain.

In [7]:
level_counts_per_domain = {
    1: Counter(),
    2: Counter(),
    3: Counter(),
    4: Counter(),
    5: Counter(),
}

for manual_assessment_cell in df["manual_assessment"]:
    assessment_list = json.loads(manual_assessment_cell)
    for assessment in assessment_list:
        for domain_number, risk_level in enumerate(assessment["domain_risk_levels"], start=1):
            level_counts_per_domain[domain_number][risk_level] += 1

column_rename_map = {
    1: "domain_1",
    2: "domain_2",
    3: "domain_3",
    4: "domain_4",
    5: "domain_5",
}

(
    pd.DataFrame(level_counts_per_domain)
    .fillna(0)
    .astype(int)
    .rename_axis("risk_level")
    .rename(columns=column_rename_map)
)

,domain_1,domain_2,domain_3,domain_4,domain_5
risk_level,,,,,
Some concerns,98,82,11,47,177
Low,133,149,218,182,60
High,14,14,15,15,8
NaN,0,0,1,1,0


## Example 5: Answer breakdown for all domain-3 signaling questions in roboto2 assessments

Domain 3 has four distinct signaling questions, keyed by the `question` text. We use `expertPrediction` as the canonical answer field.

In [8]:
answer_counts_per_question = {}  # question_text -> Counter of answers

for roboto2_assessment_cell in df["roboto2_assessment"]:
    assessment_list = json.loads(roboto2_assessment_cell)
    for assessment in assessment_list:
        for signaling_question in assessment["result"]:
            if signaling_question.get("domain_number") != 3:
                continue
            question_text = signaling_question.get("question")
            expert_answer = signaling_question.get("expertPrediction")
            if question_text not in answer_counts_per_question:
                answer_counts_per_question[question_text] = Counter()
            answer_counts_per_question[question_text][expert_answer] += 1

for question_text, answer_counts in answer_counts_per_question.items():
    print(question_text)
    for answer, count in answer_counts.most_common():
        print(f"  {answer!r}: {count}")
    print()

Were data for this outcome available for all, o rnearly all, participants randomized?
  'no': 83
  'yes': 69
  'probably yes': 58
  'probably no': 17
  'no information': 4
  'qa-na': 2

Is there evidence that the result was not biased by missing outcome data?
  'qa-na': 91
  'no': 64
  'probably yes': 50
  'yes': 12
  'probably no': 12
  'no information': 4

Could missingness in the outcome depend on its true value?
  'qa-na': 94
  'no': 59
  'probably no': 36
  'probably yes': 26
  'no information': 10
  'yes': 6
  'No': 2

Is it likely that missingness in the outcome depended on its true value?
  'qa-na': 100
  'probably no': 43
  'no': 40
  'no information': 29
  'probably yes': 18
  'yes': 1
  'No information': 1
  'No': 1



## Example 6: Overall risk-of-bias count across all assessments (both columns)

Manual assessments use `Low` / `Some concerns` / `High`; roboto2 uses `Low Risk` / `Some Concerns` / `High Risk`. Normalize to a single 3-value scale before tallying.

In [9]:
def normalize_risk(risk_level):
    # roboto2 uses "Low Risk"/"Some Concerns"/"High Risk";
    # manual uses "Low"/"Some concerns"/"High". Collapse to one scale.
    if risk_level.endswith(" Risk"):
        risk_level = risk_level[: -len(" Risk")]
    return risk_level.title()

overall_risk_counts = Counter()
for column_name in ("manual_assessment", "roboto2_assessment"):
    for assessment_cell in df[column_name]:
        assessment_list = json.loads(assessment_cell)
        for assessment in assessment_list:
            normalized_risk_level = normalize_risk(assessment["overall_risk_level"])
            overall_risk_counts[normalized_risk_level] += 1

pd.Series(overall_risk_counts, name="count").sort_values(ascending=False)

Some Concerns    282
High             132
Low               64
Name: count, dtype: int64

## Example 7: Signaling questions where gptPrediction and expertPrediction disagree

Iterate every signaling question across every roboto2 assessment. Skip entries where either field is `None`, then count disagreements among the rest. Also report how many entries were skipped because each field was `None`.

In [10]:
total_signaling_questions = 0
compared_signaling_questions = 0
disagreement_count = 0
gpt_prediction_none_count = 0
expert_prediction_none_count = 0

for roboto2_assessment_cell in df["roboto2_assessment"]:
    assessment_list = json.loads(roboto2_assessment_cell)
    for assessment in assessment_list:
        for signaling_question in assessment["result"]:
            total_signaling_questions += 1
            gpt_prediction = signaling_question.get("gptPrediction")
            expert_prediction = signaling_question.get("expertPrediction")
            if gpt_prediction is None:
                gpt_prediction_none_count += 1
            if expert_prediction is None:
                expert_prediction_none_count += 1
            if gpt_prediction is None or expert_prediction is None:
                continue
            compared_signaling_questions += 1
            if gpt_prediction != expert_prediction:
                disagreement_count += 1

if compared_signaling_questions > 0:
    disagreement_percent = disagreement_count / compared_signaling_questions * 100
else:
    disagreement_percent = 0.0

print(f"Total signaling questions:        {total_signaling_questions:,}")
print(f"gptPrediction is None:            {gpt_prediction_none_count:,}")
print(f"expertPrediction is None:         {expert_prediction_none_count:,}")
print(f"Compared (both non-None):         {compared_signaling_questions:,}")
print(f"Disagreements:                    {disagreement_count:,} ({disagreement_percent:.1f}%)")

Total signaling questions:        5,126
gptPrediction is None:            0
expertPrediction is None:         0
Compared (both non-None):         5,126
Disagreements:                    2,620 (51.1%)


## Example 8: Look up paper metadata by paper_id

The `paper_parse` column holds the S2ORC-style parse of each paper as a JSON dict (or `null` when no parse is available). This example shows how to retrieve title, abstract, and body-paragraph count for a given paper_id.

In [14]:
def get_paper_metadata(paper_id):
    matching_rows = df[df["paper_id"] == paper_id]
    if matching_rows.empty:
        return None
    paper_parse_cell = matching_rows.iloc[0]["paper_parse"]
    parse_object = json.loads(paper_parse_cell)
    return parse_object  # may be None when no S2ORC parse exists for this paper

# Pick the first paper whose parse is non-null so the example output is informative.
papers_with_parse = df[df["paper_parse"] != "null"]
example_paper_id = papers_with_parse["paper_id"].iloc[1]
parse_object = get_paper_metadata(example_paper_id)

print(f"paper_id: {example_paper_id}")
if parse_object is None:
    print("No paper parse available for this paper.")
else:
    print(f"Top-level keys: {list(parse_object.keys())}")
    title = parse_object.get("title")
    abstract = parse_object.get("abstract")
    pdf_parse = parse_object.get("pdf_parse") or {}
    body_text = pdf_parse.get("body_text") or []
    print(f"Title: {title}")
    if abstract:
        abstract_excerpt = abstract[:300]
        suffix = "..." if len(abstract) > 300 else ""
        print(f"Abstract excerpt: {abstract_excerpt}{suffix}")
    else:
        print("Abstract: (none)")
    print(f"Body paragraph count: {len(body_text)}")

paper_id: 29200
Top-level keys: ['abstract', 'authors', 'header', 'identifiers', 'paper_id', 'pdf_parse', 'title', 'venue', 'year']
Title: Epogam evening primrose oil treatment in atopic dermatitis and asthma
Abstract excerpt: Essential fatty acids are claimed to have positive effects in atopic diseases. In a double blind, placebo controlled, parallel group study 58 out of 60 children, with atopic dermatitis and the need for regular treatment with topical skin steroids, completed a 16 weeks' treatment period with either E...
Body paragraph count: 17


## Example 9: gpt-vs-expert disagreement rate per domain

Extends Example 7 by breaking the disagreement rate down by `domain_number` (1–5). Reveals which domain the model agrees with the human expert on most often.

In [15]:
per_domain_total = Counter()
per_domain_compared = Counter()
per_domain_disagree = Counter()

for roboto2_assessment_cell in df["roboto2_assessment"]:
    assessment_list = json.loads(roboto2_assessment_cell)
    for assessment in assessment_list:
        for signaling_question in assessment["result"]:
            domain_number = signaling_question.get("domain_number")
            per_domain_total[domain_number] += 1
            gpt_prediction = signaling_question.get("gptPrediction")
            expert_prediction = signaling_question.get("expertPrediction")
            if gpt_prediction is None or expert_prediction is None:
                continue
            per_domain_compared[domain_number] += 1
            if gpt_prediction != expert_prediction:
                per_domain_disagree[domain_number] += 1

per_domain_rows = []
for domain_number in sorted(per_domain_total.keys()):
    total_count = per_domain_total[domain_number]
    compared_count = per_domain_compared[domain_number]
    disagree_count = per_domain_disagree[domain_number]
    if compared_count > 0:
        disagree_percent = disagree_count / compared_count * 100
    else:
        disagree_percent = 0.0
    per_domain_rows.append({
        "domain_number":        domain_number,
        "total":                total_count,
        "compared":             compared_count,
        "disagreements":        disagree_count,
        "disagreement_percent": round(disagree_percent, 1),
    })

pd.DataFrame(per_domain_rows).set_index("domain_number")

,total,compared,disagreements,disagreement_percent
domain_number,,,,
1,699,699,331,47.4
2,1631,1631,839,51.4
3,932,932,501,53.8
4,1165,1165,591,50.7
5,699,699,358,51.2


## Example 10: Manual answer distribution per (domain, question)

Mirror of Example 5 for the `manual_assessment` column. Manual entries don't carry the question text, only `domain_number` and `question_number`, so we group by that pair and tally `expertPrediction`.

In [16]:
manual_answer_counts = {}  # (domain_number, question_number) -> Counter of answers

for manual_assessment_cell in df["manual_assessment"]:
    assessment_list = json.loads(manual_assessment_cell)
    for assessment in assessment_list:
        for signaling_question in assessment["result"]:
            domain_number = signaling_question.get("domain_number")
            question_number = signaling_question.get("question_number")
            expert_answer = signaling_question.get("expertPrediction")
            question_key = (domain_number, question_number)
            if question_key not in manual_answer_counts:
                manual_answer_counts[question_key] = Counter()
            manual_answer_counts[question_key][expert_answer] += 1

for question_key in sorted(manual_answer_counts.keys()):
    domain_number, question_number = question_key
    answer_counts = manual_answer_counts[question_key]
    print(f"Domain {domain_number}, Question {question_number}")
    for answer, count in answer_counts.most_common():
        print(f"  {answer!r}: {count}")
    print()

Domain 1, Question 1
  'Y/PY': 74
  'Y': 68
  'NI': 66
  'PY': 34
  'N': 2
  'N/PN': 1

Domain 1, Question 2
  'NI': 104
  'Y/PY': 54
  'Y': 42
  'PY': 41
  'N/PN': 2
  'PN': 1
  'N': 1

Domain 1, Question 3
  'N/PN': 77
  'PN': 64
  'N': 64
  'NI': 29
  'PY': 6
  'Y/PY': 5

Domain 2, Question 1
  'PN': 57
  'N': 45
  'Y/PY': 42
  'NI': 37
  'N/PN': 37
  'PY': 18
  'Y': 9

Domain 2, Question 2
  'PN': 53
  'Y/PY': 51
  'NI': 44
  'N': 40
  'N/PN': 24
  'PY': 19
  'Y': 14

Domain 2, Question 3
  'N/PN': 69
  'NI': 45
  None: 22
  'PN': 13
  'PY': 6
  'N': 1
  'Y/PY': 1

Domain 2, Question 4
  None: 99
  'PN': 4
  'PY': 2
  'Y/PY': 1

Domain 2, Question 5
  None: 99
  'NI': 1
  'PY': 1
  'N/PN': 1
  'PN': 1

Domain 2, Question 6
  'Y/PY': 90
  'NI': 56
  'PY': 55
  'Y': 36
  'N/PN': 5
  'PN': 3

Domain 2, Question 7
  None: 90
  'PN': 48
  'NI': 7
  'N/PN': 6
  'PY': 3

Domain 3, Question 1
  'PY': 69
  'Y/PY': 63
  'Y': 56
  'N/PN': 33
  'NI': 11
  'PN': 8
  'N': 5

Domain 3, Question 2

## Example 11: Evidence-passage count

Each roboto2 signaling question can cite evidence in two ways:
* `addedParagraphs` — paragraphs the expert added during review
* `ctxParagraph1` / `ctxParagraph2` / `ctxParagraph3` — paragraphs retrieved by the model

Count both to characterize the evidence coverage of the dataset.

In [17]:
total_added_paragraphs = 0
result_entries_with_added = 0
total_ctx_paragraphs = 0

for roboto2_assessment_cell in df["roboto2_assessment"]:
    assessment_list = json.loads(roboto2_assessment_cell)
    for assessment in assessment_list:
        for signaling_question in assessment["result"]:
            added_paragraphs = signaling_question.get("addedParagraphs") or []
            if added_paragraphs:
                result_entries_with_added += 1
            total_added_paragraphs += len(added_paragraphs)
            for context_key in ("ctxParagraph1", "ctxParagraph2", "ctxParagraph3"):
                if signaling_question.get(context_key):
                    total_ctx_paragraphs += 1

print(f"Total addedParagraphs across all result entries: {total_added_paragraphs:,}")
print(f"Result entries with >=1 added paragraph:         {result_entries_with_added:,}")
print(f"Total non-empty ctxParagraph slots:              {total_ctx_paragraphs:,}")
print(f"Sum (addedParagraphs + ctxParagraphs):           {total_added_paragraphs + total_ctx_paragraphs:,}")

Total addedParagraphs across all result entries: 154
Result entries with >=1 added paragraph:         116
Total non-empty ctxParagraph slots:              15,378
Sum (addedParagraphs + ctxParagraphs):           15,532
